<div style="background:#ffffff;border:1px solid #000000;border-radius:0.75rem;padding:24px 28px;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;">
  <div style="font-size:0.85em;letter-spacing:0.12em;text-transform:uppercase;color:#6e7781;">Stanford Musculoskeletal Imaging Workshop · November 2026</div>
  <h1 style="margin:6px 0 4px;font-size:2.2em;color:#111827;font-weight:700;"><span style="color:#f97316;">MuscleMap</span> hands-on</h1>
  <div style="font-size:1.15em;color:#5c5962;">Segmenting a lumbar spine MRI, from image to muscle size and fat</div>
  <img src="../assets/v1.4_figure.gif" alt="MuscleMap whole-body muscle segmentation" style="display:block;width:100%;border-radius:0.5rem;margin:18px 0 4px;">
  <table style="margin-top:20px;border-collapse:collapse;width:100%;color:inherit;font-size:0.95em;background:transparent;">
    <tr style="background:transparent;color:#6e7781;text-align:left;"><th style="padding:6px 8px;border-bottom:2px solid #f97316;">Step</th><th style="padding:6px 8px;border-bottom:2px solid #f97316;">What happens</th><th style="padding:6px 8px;border-bottom:2px solid #f97316;">Time</th></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">1 · Load</td><td style="padding:6px 8px;color:inherit;">Open the example MRI and view the four slices</td><td style="padding:6px 8px;color:inherit;">1 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">2 · Segment</td><td style="padding:6px 8px;color:inherit;">Run <code>mm_segment</code> and read its log</td><td style="padding:6px 8px;color:inherit;">2–10 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">3 · Inspect</td><td style="padding:6px 8px;color:inherit;">Interactive overlay viewer</td><td style="padding:6px 8px;color:inherit;">2 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">4 · Measure</td><td style="padding:6px 8px;color:inherit;">Cross-sectional area of psoas, multifidus and erector spinae</td><td style="padding:6px 8px;color:inherit;">1 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">5 · Quantify fat</td><td style="padding:6px 8px;color:inherit;">Run <code>mm_extract_metrics</code> with a Gaussian mixture model</td><td style="padding:6px 8px;color:inherit;">2–5 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">6 · Fat maps</td><td style="padding:6px 8px;color:inherit;">Fat overlay and fat percentage per muscle</td><td style="padding:6px 8px;color:inherit;">2 min</td></tr>
    <tr style="background:transparent;"><td style="padding:6px 8px;color:inherit;">Example 2</td><td style="padding:6px 8px;color:inherit;">Thigh Dixon MRI: segment both thighs on the fat-fraction map, volume and fat fraction per muscle</td><td style="padding:6px 8px;color:inherit;">5 min</td></tr>
  </table>
</div>

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;"><b>Kernel.</b> The top-right corner of this notebook must say <b>Python (MuscleMap)</b>. If it does not, choose it under <i>Kernel → Change Kernel</i>. The setup instructions are in the README of this repository.</div>

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;"><b>First run.</b> The first time <code>mm_segment</code> runs it downloads the model weights (about 100 MB). Do this at home, not on conference Wi-Fi.</div>

Run the cells one by one with **Shift + Enter**.

## Setup

In [ ]:
import io
import json
import os
import shutil
import urllib.request
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import nibabel as nib
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, HTML

pio.templates.default = "plotly_white"
CLEAR = "#ffffff"                                        # figure background
SLIDER = dict(font=dict(color="#333"), bgcolor="#eeeeee", bordercolor="#dddddd", tickcolor="#999999")

# Locate the repository root (the folder that contains data/MuscleMap), starting from this notebook
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "MuscleMap").is_dir())
image_path = repo_root / "data" / "MuscleMap" / "source-data" / "lumbar" / "sub-01_T2w_L4-L5.nii.gz"
thigh_image_path = repo_root / "data" / "MuscleMap" / "source-data" / "thigh" / "sub-workshop_thigh-example.nii.gz"
outputs_root = repo_root / "data" / "MuscleMap" / "outputs"   # everything MuscleMap produces goes here
lumbar_output_dir = outputs_root / "lumbar"                     # Example 1
thigh_output_dir = outputs_root / "thigh"                       # Example 2
for d in (lumbar_output_dir, thigh_output_dir):
    d.mkdir(parents=True, exist_ok=True)

# mm_segment lives next to the kernel's Python: <env>/Scripts on Windows, <env>/bin on macOS and Linux
env_dir = Path(sys.executable).parent
candidates = [env_dir / "Scripts" / "mm_segment.exe", env_dir / "mm_segment", shutil.which("mm_segment")]
mm_segment = next((str(c) for c in candidates if c and Path(c).exists()), None)
mm_extract_metrics = str(Path(mm_segment).with_name(Path(mm_segment).name.replace("mm_segment", "mm_extract_metrics"))) if mm_segment else None

def status_table(rows):
    """rows: list of (label, ok, text)"""
    dot = lambda ok: f'<span style="color:{"#3ddc84" if ok else "#ff5c5c"};font-size:1.1em;">●</span>'
    html = '<table style="border-collapse:collapse;font-family:ui-monospace,Menlo,Consolas,monospace;font-size:0.9em;">'
    for label, ok, text in rows:
        html += f'<tr><td style="padding:4px 10px 4px 0;color:#8a8a93;">{label}</td><td style="padding:4px 8px;">{dot(ok)}</td><td style="padding:4px 0;">{text}</td></tr>'
    display(HTML(html + "</table>"))

status_table([
    ("Kernel",     "MuscleMap" in sys.executable, sys.executable),
    ("mm_segment", mm_segment is not None,        mm_segment or "not found: is MuscleMap installed in this environment?"),
    ("Lumbar image", image_path.exists(),         str(image_path)),
    ("Thigh image",  thigh_image_path.exists(),   str(thigh_image_path)),
    ("Outputs",    outputs_root.is_dir(),         str(outputs_root) + "  (lumbar/, thigh/)"),
])

Every structure gets one colour that is used in the table, the overlay and the 3D view. Left and right of the same muscle share a hue; the right side is the lighter shade.

In [ ]:
LABELS = {
    1101: "Left Levator Scapulae",
    1102: "Right Levator Scapulae",
    1111: "Left Semispinalis Cervicis and Multifidus",
    1112: "Right Semispinalis Cervicis and Multifidus",
    1121: "Left Semispinalis Capitis",
    1122: "Right Semispinalis Capitis",
    1131: "Left Splenius Capitis",
    1132: "Right Splenius Capitis",
    1141: "Left Sternocleidomastoid",
    1142: "Right Sternocleidomastoid",
    1151: "Left Longus Colli",
    1152: "Right Longus Colli",
    1161: "Left Trapezius",
    1162: "Right Trapezius",
    2101: "Left Supraspinatus",
    2102: "Right Supraspinatus",
    2111: "Left Subscapularis",
    2112: "Right Subscapularis",
    2121: "Left Infraspinatus",
    2122: "Right Infraspinatus",
    2141: "Left Deltoid",
    2142: "Right Deltoid",
    4101: "Left Rhomboid",
    4102: "Right Rhomboid",
    5101: "Left Thoracolumbar Multifidus",
    5102: "Right Thoracolumbar Multifidus",
    5111: "Left Erector Spinae",
    5112: "Right Erector Spinae",
    5121: "Left Psoas Major",
    5122: "Right Psoas Major",
    5131: "Left Quadratus Lumborum",
    5132: "Right Quadratus Lumborum",
    5141: "Left Latissimus Dorsi",
    5142: "Right Latissimus Dorsi",
    6101: "Left Gluteus Minimus",
    6102: "Right Gluteus Minimus",
    6111: "Left Gluteus Medius",
    6112: "Right Gluteus Medius",
    6121: "Left Gluteus Maximus",
    6122: "Right Gluteus Maximus",
    6131: "Left Tensor Fasciae Latae",
    6132: "Right Tensor Fasciae Latae",
    6141: "Left Iliacus",
    6142: "Right Iliacus",
    6151: "Left Ilium",
    6152: "Right Ilium",
    6160: "Sacrum",
    6171: "Left Femur",
    6172: "Right Femur",
    6181: "Left Piriformis",
    6182: "Right Piriformis",
    6191: "Left Pectineus",
    6192: "Right Pectineus",
    6201: "Left Obturator Internus",
    6202: "Right Obturator Internus",
    6211: "Left Obturator Externus",
    6212: "Right Obturator Externus",
    6221: "Left Gemelli and Quadratus Femoris",
    6222: "Right Gemelli and Quadratus Femoris",
    7101: "Left Vastus Lateralis",
    7102: "Right Vastus Lateralis",
    7111: "Left Vastus Intermedius",
    7112: "Right Vastus Intermedius",
    7121: "Left Vastus Medialis",
    7122: "Right Vastus Medialis",
    7131: "Left Rectus Femoris",
    7132: "Right Rectus Femoris",
    7141: "Left Sartorius",
    7142: "Right Sartorius",
    7151: "Left Gracilis",
    7152: "Right Gracilis",
    7161: "Left Semimembranosus",
    7162: "Right Semimembranosus",
    7171: "Left Semitendinosus",
    7172: "Right Semitendinosus",
    7181: "Left Biceps Femoris Long Head",
    7182: "Right Biceps Femoris Long Head",
    7191: "Left Biceps Femoris Short Head",
    7192: "Right Biceps Femoris Short Head",
    7201: "Left Adductor Magnus",
    7202: "Right Adductor Magnus",
    7211: "Left Adductor Longus",
    7212: "Right Adductor Longus",
    7221: "Left Adductor Brevis",
    7222: "Right Adductor Brevis",
    7231: "Left Patella",
    7232: "Right Patella",
    8101: "Left Tibialis Anterior",
    8102: "Right Tibialis Anterior",
    8111: "Left Tibialis Posterior",
    8112: "Right Tibialis Posterior",
    8121: "Left Peroneus Longus",
    8122: "Right Peroneus Longus",
    8131: "Left Soleus",
    8132: "Right Soleus",
    8141: "Left Medial Gastrocnemius",
    8142: "Right Medial Gastrocnemius",
    8151: "Left Lateral Gastrocnemius",
    8152: "Right Lateral Gastrocnemius",
    8161: "Left Tibia",
    8162: "Right Tibia",
    8171: "Left Fibula",
    8172: "Right Fibula",
    8181: "Left Flexor Hallucis Longus",
    8182: "Right Flexor Hallucis Longus",
    8191: "Left Extensor Digitorum Longus",
    8192: "Right Extensor Digitorum Longus",
    8201: "Left Flexor Digitorum Longus",
    8202: "Right Flexor Digitorum Longus",
    8211: "Left Popliteus",
    8212: "Right Popliteus",
    8221: "Left Plantaris",
    8222: "Right Plantaris",
}

PALETTE = ["#4F8EF7", "#F28E2B", "#E15759", "#59A14F", "#B07AA1", "#EDC948",
           "#76B7B2", "#FF9DA7", "#9C755F", "#17BECF", "#D4A017", "#8CD17D",
           "#F1CE63", "#A0CBE8", "#FFBE7D", "#D37295", "#B6992D", "#499894"]

def _lighten(hex_color, amount=0.38):
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (1, 3, 5))
    r, g, b = (int(c + (255 - c) * amount) for c in (r, g, b))
    return f"#{r:02X}{g:02X}{b:02X}"

def color_for(label, _cache={}):
    """Same hue for left (…1) and right (…2) of a muscle; unpaired labels get their own colour."""
    muscle = label // 10
    if muscle not in _cache:
        _cache[muscle] = PALETTE[len(_cache) % len(PALETTE)]
    base = _cache[muscle]
    return _lighten(base) if label % 10 == 2 else base

## 1 · Load and view the image

The example is an axial T2-weighted MRI of the lumbar spine at the L4–L5 disc: four slices of 4.4 mm, 0.69 mm in-plane. The image is shown in radiological convention: the patient's right is on the left of the image, anterior is at the top.

In [ ]:
img = nib.load(image_path)
data = img.get_fdata()
voxel = tuple(float(v) for v in img.header.get_zooms())
n_slices = data.shape[2]

display(pd.DataFrame({
    "value": [str(data.shape), " × ".join(f"{v:.2f}" for v in voxel) + " mm",
              "".join(nib.aff2axcodes(img.affine)), f"{np.prod(voxel) / 1000:.3f} ml"],
}, index=["shape (x, y, z)", "voxel size", "orientation", "voxel volume"]).style.set_properties(**{"text-align": "left"}))

# Grey-level window for display, shared by all views
lo, hi = np.percentile(data, [1, 99.5])

def to_display(a):
    """Reorient a (x, y) slice for display: anterior at the top, patient's right on the left."""
    return np.flipud(a.T)

# Crop to the body (plus a margin) so the panels spend their pixels on anatomy rather than air.
# Rows and columns whose mean intensity is below 10 % of the window maximum are air.
mean_slice = to_display(data.mean(axis=2))
rows_ = np.where(mean_slice.mean(axis=1) > 0.10 * hi)[0]
cols_ = np.where(mean_slice.mean(axis=0) > 0.10 * hi)[0]
pad = 8
crop = (slice(max(rows_.min() - pad, 0), rows_.max() + pad), slice(max(cols_.min() - pad, 0), cols_.max() + pad))

crop_rows = crop[0].stop - crop[0].start
crop_cols = crop[1].stop - crop[1].start
def panel_height(n_cols, fig_width=1000, spacing=12):
    """Pixel height of one image panel when n_cols cropped slices share the figure width."""
    return (fig_width - spacing * (n_cols - 1)) / n_cols * crop_rows / crop_cols

def to_rgb(k):
    """Slice k as a cropped RGB uint8 array."""
    g = np.clip((to_display(data[:, :, k])[crop] - lo) / (hi - lo), 0, 1)
    g = (g * 255).astype(np.uint8)
    return np.stack([g, g, g], axis=-1)

GRID_COLS = 2                                        # slices per row: 2 keeps each panel large enough for full resolution

def slice_grid(frames, title=None, extra_sliders=(), cols=GRID_COLS):
    """One panel per axial slice in a grid. frames: list (per slice) of trace lists."""
    rows = -(-len(frames) // cols)
    fig = make_subplots(rows=rows, cols=cols, horizontal_spacing=0.015, vertical_spacing=0.07,
                        subplot_titles=[f"slice {k}" for k in range(len(frames))])
    for k, traces in enumerate(frames):
        r, c = divmod(k, cols)
        for t in traces:
            fig.add_trace(t, row=r + 1, col=c + 1)
        fig.update_xaxes(visible=False, row=r + 1, col=c + 1)
        fig.update_yaxes(visible=False, scaleanchor=f"x{k + 1 if k else ''}", row=r + 1, col=c + 1)
    top = 60 if title else 36
    ph = panel_height(cols)
    fig.update_layout(title=title, height=int(top + rows * ph + (rows - 1) * 0.07 * rows * ph + 20 + 80 * len(extra_sliders)),
                      margin=dict(l=10, r=10, t=top, b=10 + 80 * len(extra_sliders)),
                      paper_bgcolor=CLEAR, plot_bgcolor=CLEAR, sliders=list(extra_sliders))
    return fig

slice_grid([[go.Image(z=to_rgb(k), hoverinfo="skip")] for k in range(n_slices)]).show()

## 2 · Run MuscleMap

`mm_segment` takes the image (`-i`), the model (`-r`) and an output folder (`-o`). The whole-body model segments 113 muscles and bones and is contrast-agnostic. The output, `sub-01_T2w_L4-L5_dseg.nii.gz` plus a JSON sidecar with the model version and settings, is written to `data/MuscleMap/outputs/lumbar/`.

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;">MuscleMap uses the GPU when it finds one and the CPU otherwise. On a CPU this image takes a few minutes. The MuscleMap log streams into the notebook while it runs; the summary underneath shows which device was used.</div>

In [ ]:
command = [mm_segment, "-i", str(image_path), "-r", "wholebody", "-o", str(lumbar_output_dir)]
display(HTML('<div style="font-family:ui-monospace,Menlo,Consolas,monospace;background:#17171a;border-radius:8px;padding:10px 14px;color:#e6e6e6;">'
             '<span style="color:#8a8a93;">$</span> mm_segment ' + " ".join(command[1:]) + "</div>"))

env = {**os.environ, "MPLBACKEND": "Agg"}             # Jupyter's inline plotting backend must not leak into mm_segment

def log_box(lines, title):
    """The MuscleMap log as an expanded, scrollable block."""
    body = "\n".join(lines).replace("<", "&lt;") or "..."
    return HTML(f'<details open style="margin-top:8px;"><summary style="cursor:pointer;color:#6e7781;">{title}</summary>'
                f'<pre style="background:#0f0f10;color:#cfcfd4;padding:12px;border-radius:8px;font-size:0.8em;max-height:420px;overflow:auto;">{body}</pre></details>')

log_lines, t0 = [], time.time()
live = display(log_box([], "MuscleMap log - starting"), display_id=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in process.stdout:                          # stream the log into the notebook while MuscleMap runs
    log_lines.append(line.rstrip("\n"))
    live.update(log_box(log_lines, f"MuscleMap log - running {time.time() - t0:.0f} s"))
process.wait()
duration = time.time() - t0

seg_path = lumbar_output_dir / image_path.name.replace(".nii.gz", "_dseg.nii.gz")
json_path = seg_path.with_name(seg_path.name.replace(".nii.gz", ".json"))
info = json.loads(json_path.read_text()) if json_path.exists() else {}
ok = process.returncode == 0 and seg_path.exists()

summary = [("Result", ok, "segmentation written" if ok else f"failed (exit code {process.returncode})"),
           ("Duration", True, f"{duration:.0f} s"),
           ("Device", True, str(info.get("device", "?")).upper()),
           ("Model", True, f'{info.get("model", "?")}  v{info.get("model_version", "?")}'),
           ("Output", seg_path.exists(), str(seg_path))]
live.update(log_box(log_lines, f"MuscleMap log - {len(log_lines)} lines - {duration:.0f} s"))
status_table(summary)

## 3 · Overlay viewer

The whole-body model labels 14 structures in this image. From here on we work with three muscles only: **psoas major**, **multifidus** and **erector spinae**, left and right. The cell below keeps those six labels, saves them as a second segmentation file (`…_dseg-paraspinal.nii.gz`) that the later steps use, and shows them on top of the image. Change the transparency with the slider and hover over a muscle to see its name.

In [ ]:
seg_img = nib.load(seg_path)
seg_all = seg_img.get_fdata().astype(int)
assert seg_all.shape == data.shape, "segmentation and image have different shapes"

TARGETS = ["Psoas Major", "Multifidus", "Erector Spinae"]
SHORT_NAMES = {"Thoracolumbar Multifidus": "Multifidus"}     # MuscleMap label name -> name used in this notebook

def structure_of(label):
    name = LABELS.get(label, str(label)).replace("Left ", "").replace("Right ", "")
    return SHORT_NAMES.get(name, name)

def display_name(label):
    side = "Left " if label % 10 == 1 else "Right " if label % 10 == 2 else ""
    return side + structure_of(label)

all_labels = [int(v) for v in np.unique(seg_all) if v != 0]
present = [v for v in all_labels if structure_of(v) in TARGETS]
seg = np.where(np.isin(seg_all, present), seg_all, 0)          # the three muscles only
colors = {v: color_for(v) for v in present}

seg_sel_path = lumbar_output_dir / image_path.name.replace(".nii.gz", "_dseg-paraspinal.nii.gz")
nib.save(nib.Nifti1Image(seg.astype(np.int16), seg_img.affine, seg_img.header), seg_sel_path)
print(f"{len(all_labels)} structures segmented, {len(present)} kept:", ", ".join(display_name(v) for v in present))
print("saved", seg_sel_path.name)

def overlay_rgba(k, label_names):
    """Segmentation of slice k as an RGBA uint8 array (transparent where there is no label)."""
    s = to_display(seg[:, :, k])[crop]
    rgba = np.zeros(s.shape + (4,), dtype=np.uint8)
    names = np.full(s.shape, "", dtype=object)
    for v, c in colors.items():
        m = s == v
        rgba[m] = [int(c[i:i + 2], 16) for i in (1, 3, 5)] + [255]
        names[m] = label_names[v]
    return rgba, names

frames = []
for k in range(n_slices):
    rgba, names = overlay_rgba(k, {v: display_name(v) for v in present})
    frames.append([go.Image(z=to_rgb(k), hoverinfo="skip"),
                   go.Image(z=rgba, colormodel="rgba", opacity=0.5, text=names,
                            hovertemplate="%{text}<extra></extra>")])

opacity_steps = [dict(method="restyle", label=f"{a:.1f}",
                      args=[{"opacity": a}, [2 * k + 1 for k in range(n_slices)]]) for a in np.arange(0, 1.01, 0.1)]
opacity_slider = dict(active=5, steps=opacity_steps, currentvalue=dict(prefix="overlay opacity "), pad=dict(t=10),
                      **SLIDER)

slice_grid(frames, None, [opacity_slider]).show()

## 4 · Volume and cross-sectional area

For every slice the cross-sectional area (CSA) is the number of labelled pixels times the pixel area. From that follow three measures per muscle and side:

* **CSA (mm²)**: the total cross-sectional area, summed over all slices.
* **aCSA (mm²)**: the average cross-sectional area per slice, CSA divided by the number of slices.
* **Volume (ml)**: CSA times the slice thickness.

In [ ]:
pixel_mm2 = voxel[0] * voxel[1]                     # in-plane pixel area
slice_mm = voxel[2]                                  # slice thickness

# Order the slices from cranial to caudal using the scanner coordinate of each slice centre
z_mm = np.array([(img.affine @ [data.shape[0] / 2, data.shape[1] / 2, k, 1])[2] for k in range(n_slices)])
order = np.argsort(-z_mm)                            # highest (most cranial) first

rows = []
for v in present:
    name = LABELS.get(v, str(v))
    structure = structure_of(v)
    csa = np.array([(seg[:, :, k] == v).sum() * pixel_mm2 for k in range(n_slices)])
    rows.append(dict(structure=structure, side="Left" if name.startswith("Left ") else "Right", label=v,
                     **{"volume (ml)": csa.sum() * slice_mm / 1000, "CSA (mm²)": csa.sum(), "aCSA (mm²)": csa.mean()},
                     csa_cranial_to_caudal=csa[order]))
muscles = pd.DataFrame(rows).set_index(["structure", "side"]).reindex(TARGETS, level=0)

table = muscles[["volume (ml)", "CSA (mm²)", "aCSA (mm²)"]]
styled = (table.style
          .format({"volume (ml)": "{:.1f}", "CSA (mm²)": "{:.0f}", "aCSA (mm²)": "{:.0f}"})
          .apply(lambda col: [f"border-left: 10px solid {colors[muscles.loc[i, 'label']]}" for i in col.index], subset=["volume (ml)"])
          .set_caption(f"{n_slices} slices of {slice_mm:.1f} mm at L4–L5 · CSA summed over the slices · aCSA per slice")
          .set_table_styles([{"selector": "caption", "props": "caption-side: top; color: #6e7781; padding-bottom: 6px;"}]))
display(styled)

### CSA along the muscle, normalised to 0–100 %

As in [Stewart et al. 2024](https://pmc.ncbi.nlm.nih.gov/articles/PMC11634488/), the per-slice values are expressed along the length of the imaged muscle: the most superior (cranial) slice is 0 %, the most inferior (caudal) slice 100 %, and the values in between are interpolated with a spline to 101 points. One panel per muscle; solid is left, dashed is right, the dots are the measured slices.

In [ ]:
try:
    from scipy.interpolate import CubicSpline
    def resample(y, x_new):
        return CubicSpline(np.linspace(0, 100, len(y)), y)(x_new)
except ImportError:                                  # fall back to linear interpolation
    def resample(y, x_new):
        return np.interp(x_new, np.linspace(0, 100, len(y)), y)

length_pct = np.linspace(0, 100, 101)
x_slices = np.linspace(0, 100, n_slices)

fig = make_subplots(rows=1, cols=len(TARGETS), subplot_titles=TARGETS, shared_yaxes=True, horizontal_spacing=0.09)
for col, structure in enumerate(TARGETS, start=1):
    for side, dash in (("Left", "solid"), ("Right", "dash")):
        if (structure, side) not in muscles.index:
            continue
        y = muscles.loc[(structure, side), "csa_cranial_to_caudal"]
        c = colors[muscles.loc[(structure, side), "label"]]
        fig.add_trace(go.Scatter(x=length_pct, y=resample(y, length_pct), mode="lines", name=side, legendgroup=side,
                                 showlegend=(col == 1), line=dict(color=c, width=3, dash=dash),
                                 hovertemplate="%{x:.0f} %<br>%{y:.0f} mm²<extra>" + f"{side} {structure}</extra>"),
                      row=1, col=col)
        fig.add_trace(go.Scatter(x=x_slices, y=y, mode="markers", showlegend=False, legendgroup=side,
                                 marker=dict(color=c, size=9, line=dict(color="white", width=1)), hoverinfo="skip"),
                      row=1, col=col)
    fig.update_xaxes(title_text="Length (%)", range=[0, 100], tickvals=[0, 50, 100],
                     ticktext=["superior", "50 %", "inferior"], gridcolor="#e5e7eb", row=1, col=col)
fig.update_yaxes(title_text="CSA (mm²)", rangemode="tozero", gridcolor="#e5e7eb", row=1, col=1)
fig.update_layout(height=430, paper_bgcolor=CLEAR, plot_bgcolor=CLEAR, margin=dict(l=10, r=10, t=60, b=10),
                  legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", y=-0.22, x=0.5, xanchor="center"))
fig.show()

## 5 · Fat infiltration with mm_extract_metrics

`mm_extract_metrics` quantifies each segmented muscle. On a T1- or T2-weighted image it clusters the voxel intensities inside every muscle into **two components**: muscle (dark) and fat (bright). We use a **Gaussian mixture model** (`-m gmm`); k-means (`-m kmeans`) is the alternative. The run writes a CSV with the results, NIfTI maps of both components, and the posterior probability of fat per voxel. We pass the filtered segmentation from step 3, so only the six paraspinal labels are analysed.

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;">The clustering runs separately for each label with 20 restarts, so expect a minute or two. Use <code>--qc</code> on the command line to inspect and adjust the thresholds interactively.</div>

In [ ]:
cmd = [mm_extract_metrics, "-m", "gmm", "-r", "wholebody", "-i", str(image_path), "-s", str(seg_sel_path), "-c", "2", "-o", str(lumbar_output_dir)]
display(HTML('<div style="font-family:ui-monospace,Menlo,Consolas,monospace;background:#17171a;border-radius:8px;padding:10px 14px;color:#e6e6e6;">'
             '<span style="color:#8a8a93;">$</span> mm_extract_metrics ' + " ".join(cmd[1:]) + "</div>"))

lines, t0 = [], time.time()
live = display(log_box([], "MuscleMap log - starting"), display_id=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in proc.stdout:
    lines.append(line.rstrip("\n"))
    live.update(log_box(lines, f"MuscleMap log - running {time.time() - t0:.0f} s"))
proc.wait()
live.update(log_box(lines, f"MuscleMap log - {len(lines)} lines - {time.time() - t0:.0f} s"))

stem = image_path.name.replace(".nii.gz", "")
gmm = dict(csv=lumbar_output_dir / f"{stem}_gmm_2component_results.csv",
           fat=lumbar_output_dir / f"{stem}_gmm_2component_fat_seg.nii.gz",
           softseg=lumbar_output_dir / f"{stem}_gmm_fat_2component_softseg.nii.gz")
status_table([("GMM", proc.returncode == 0, f"exit code {proc.returncode}"),
              ("Results", gmm["csv"].exists(), gmm["csv"].name),
              ("Fat map", gmm["fat"].exists(), gmm["fat"].name),
              ("Posterior", gmm["softseg"].exists(), gmm["softseg"].name)])

## 6 · Fat maps and fat percentage

The fat voxels found by the GMM in orange on top of the image; the slider sets the transparency. Then the posterior probability of fat as a heatmap, and finally the fat percentage and fat volume of the three muscles, read from the CSV.

In [ ]:
fat_map = nib.load(gmm["fat"]).get_fdata() > 0
FAT_RGB = [249, 115, 22]                              # MuscleMap orange

def fat_rgba(mask2d):
    rgba = np.zeros(mask2d.shape + (4,), dtype=np.uint8)
    rgba[mask2d] = FAT_RGB + [255]
    return rgba

frames = [[go.Image(z=to_rgb(k), hoverinfo="skip"),
           go.Image(z=fat_rgba(to_display(fat_map[:, :, k])[crop]), colormodel="rgba", opacity=0.7, hoverinfo="skip")]
          for k in range(n_slices)]
fat_slider = dict(active=7, currentvalue=dict(prefix="fat opacity "), pad=dict(t=10), **SLIDER,
                  steps=[dict(method="restyle", label=f"{a:.1f}", args=[{"opacity": a}, [2 * k + 1 for k in range(n_slices)]])
                         for a in np.arange(0, 1.01, 0.1)])
slice_grid(frames, None, [fat_slider]).show()

### GMM posterior probability of fat

The GMM does not only assign each voxel to a component, it also gives the probability that a voxel belongs to the fat component. MuscleMap writes this as a soft segmentation (`…_gmm_fat_2component_softseg.nii.gz`). Shown here as a heatmap on the image: dark means almost certainly muscle, bright yellow means almost certainly fat, and the intermediate colours mark the voxels where the decision is uncertain.

**Why the intermediate values matter: partial volume.** A voxel in this image is 0.69 × 0.69 × 4.4 mm. Fat infiltration in muscle is often finer than that: thin streaks of fat between fascicles, and the boundary between a fat deposit and the surrounding muscle, run *through* voxels rather than between them. Such a voxel contains both tissues and its signal is a mixture, somewhere between the intensity of pure muscle and pure fat. This is the partial volume effect, and it is strongest along the 4.4 mm slice direction, where one voxel averages over a lot of tissue.

A hard threshold has to call each of these mixed voxels either muscle or fat, so it over- or underestimates the fat depending on where the threshold falls. The posterior probability keeps the information: a voxel with P(fat) = 0.5 is best read as roughly half fat, half muscle. Summing the probabilities over a muscle therefore gives a fat fraction that is less sensitive to the exact threshold than counting voxels in the fat map above. Look at the edges of the bright regions in the heatmap: the orange and red rim around each yellow core is the partial volume zone.

In [ ]:
prob = nib.load(gmm["softseg"]).get_fdata().astype(float)
prob[seg == 0] = np.nan                              # transparent outside the three muscles

rows = -(-n_slices // GRID_COLS)
fig = make_subplots(rows=rows, cols=GRID_COLS, horizontal_spacing=0.015, vertical_spacing=0.07,
                    subplot_titles=[f"slice {k}" for k in range(n_slices)])
for k in range(n_slices):
    r, c = divmod(k, GRID_COLS)
    fig.add_trace(go.Image(z=to_rgb(k), hoverinfo="skip"), row=r + 1, col=c + 1)
    fig.add_trace(go.Heatmap(z=to_display(prob[:, :, k])[crop], zmin=0, zmax=1, colorscale="Inferno", opacity=0.85,
                             showscale=(k == n_slices - 1), hoverongaps=False,
                             colorbar=dict(title="P(fat)", thickness=14, len=0.8, tickvals=[0, 0.25, 0.5, 0.75, 1]),
                             hovertemplate="P(fat) = %{z:.2f}<extra></extra>"), row=r + 1, col=c + 1)
    fig.update_xaxes(visible=False, row=r + 1, col=c + 1)
    fig.update_yaxes(visible=False, scaleanchor=f"x{k + 1 if k else ''}", row=r + 1, col=c + 1)
ph = panel_height(GRID_COLS, fig_width=940)
fig.update_layout(height=int(36 + rows * ph * 1.1 + 30), margin=dict(l=10, r=70, t=36, b=10),
                  paper_bgcolor=CLEAR, plot_bgcolor=CLEAR)
fig.show()

In [ ]:
results = pd.read_csv(gmm["csv"]).set_index("Label")
fat_table = results.loc[muscles["label"].values, ["Fat (%)", "Fat volume (ml)", "Total volume (ml)"]]
fat_table.index = muscles.index

styled = (fat_table.style
          .format("{:.1f}")
          .bar(subset=["Fat (%)"], color="#f9731666", vmin=0, vmax=max(50, float(fat_table["Fat (%)"].max())))
          .apply(lambda col: [f"border-left: 10px solid {colors[muscles.loc[i, 'label']]}" for i in col.index], subset=["Fat (%)"])
          .set_caption("Fat fraction and fat volume per muscle · 2-component GMM")
          .set_table_styles([{"selector": "caption", "props": "caption-side: top; color: #6e7781; padding-bottom: 6px;"}]))
display(styled)

### Fat along the muscle, normalised to 0–100 %

The same layout as the CSA plot, now for fat: per slice, the fat voxels divided by all voxels of the muscle. Superior is 0 %, inferior 100 %, cubic-spline interpolation to 101 points. Solid is left, dashed is right, dots are the measured slices.

In [ ]:
fat_pct_per_slice = {}
for v in present:
    lab = seg == v
    fat_pct_per_slice[v] = np.array([100 * (fat_map[:, :, k] & lab[:, :, k]).sum() / max(lab[:, :, k].sum(), 1)
                                     for k in range(n_slices)])[order]

fig = make_subplots(rows=1, cols=len(TARGETS), subplot_titles=TARGETS, shared_yaxes=True, horizontal_spacing=0.09)
for col, structure in enumerate(TARGETS, start=1):
    for side, dash in (("Left", "solid"), ("Right", "dash")):
        if (structure, side) not in muscles.index:
            continue
        v = muscles.loc[(structure, side), "label"]
        y = fat_pct_per_slice[v]
        c = colors[v]
        fig.add_trace(go.Scatter(x=length_pct, y=np.clip(resample(y, length_pct), 0, 100), mode="lines", name=side,
                                 legendgroup=side, showlegend=(col == 1), line=dict(color=c, width=3, dash=dash),
                                 hovertemplate="%{x:.0f} %<br>fat %{y:.1f} %<extra>" + f"{side} {structure}</extra>"),
                      row=1, col=col)
        fig.add_trace(go.Scatter(x=x_slices, y=y, mode="markers", showlegend=False, legendgroup=side,
                                 marker=dict(color=c, size=9, line=dict(color="white", width=1)), hoverinfo="skip"),
                      row=1, col=col)
    fig.update_xaxes(title_text="Length (%)", range=[0, 100], tickvals=[0, 50, 100],
                     ticktext=["superior", "50 %", "inferior"], gridcolor="#e5e7eb", row=1, col=col)
fig.update_yaxes(title_text="Fat (%)", range=[0, 100], gridcolor="#e5e7eb", row=1, col=1)
fig.update_layout(height=430, paper_bgcolor=CLEAR, plot_bgcolor=CLEAR, margin=dict(l=10, r=10, t=50, b=10),
                  legend=dict(bgcolor="rgba(0,0,0,0)", orientation="h", y=-0.22, x=0.5, xanchor="center"))
fig.show()

### Where in the muscle is the fat? Six tiles per muscle

Fat infiltration is rarely uniform across a muscle: in the paraspinals it is typically higher posteriorly and medially. To see that, every muscle on every slice is divided into **six tiles**. One vertical line splits the muscle into a medial and a lateral half with equal numbers of voxels; two horizontal lines split it into an anterior, middle and posterior third, again with equal numbers of voxels. Both cuts follow the muscle itself, not a bounding box, so each tile holds close to one sixth of the muscle on that slice, whatever its shape. In each tile the GMM fat voxels are counted and divided by the voxels of the tile: the fat percentage of that tile.

First the tiles drawn on the middle slice, then the six tiles per muscle with all slices pooled, then tile by tile along the muscle, cranial to caudal.

In [ ]:
N_AP, N_ML = 3, 2                                      # bands anterior->posterior, halves medial->lateral
AP_NAMES = ["anterior", "middle", "posterior"]
ML_NAMES = ["medial", "lateral"]
TILE_NAMES = [f"{a[:3]}·{m[:3]}" for a in AP_NAMES for m in ML_NAMES]
TILE_COLORS = ["#4F8EF7", "#F28E2B", "#59A14F", "#E15759", "#B07AA1", "#EDC948"]

def tile_index(mask2d, side):
    """Tile number 0..5 for every voxel of a display-space mask, -1 outside.
    One vertical line splits the muscle into a medial and a lateral half with equal numbers of voxels;
    two horizontal lines split it into an anterior, middle and posterior third with equal numbers of voxels.
    Both cuts run straight through the whole muscle, so the six tiles are close to, not exactly, one sixth each."""
    idx = np.full(mask2d.shape, -1, dtype=int)
    rr, cc = np.nonzero(mask2d)
    n = rr.size
    if n == 0:
        return idx
    half = np.minimum(np.argsort(np.argsort(cc, kind="stable"), kind="stable") * N_ML // n, N_ML - 1)   # rank by column
    if side != "Left":                                   # patient's right is on the viewer's left, so medial flips
        half = N_ML - 1 - half
    band = np.minimum(np.argsort(np.argsort(rr, kind="stable"), kind="stable") * N_AP // n, N_AP - 1)   # rank by row
    idx[rr, cc] = band * N_ML + half
    return idx

def tile_counts(mask2d, fat2d, side):
    """(muscle voxels, fat voxels) per tile, shape (N_AP, N_ML)."""
    idx = tile_index(mask2d, side)
    total = np.array([(idx == t).sum() for t in range(N_AP * N_ML)], dtype=float).reshape(N_AP, N_ML)
    fat = np.array([((idx == t) & fat2d).sum() for t in range(N_AP * N_ML)], dtype=float).reshape(N_AP, N_ML)
    total[total == 0] = np.nan
    return total, fat

side_of = {v: ("Left" if display_name(v).startswith("Left") else "Right") for v in present}

# Counts per muscle, per slice (cranial -> caudal), per tile
tiles = {}                                           # label -> dict(total, fat) with shape (n_slices, N_AP, N_ML)
for v in present:
    tot = np.zeros((n_slices, N_AP, N_ML)); ft = np.zeros((n_slices, N_AP, N_ML))
    for i, k in enumerate(order):
        tot[i], ft[i] = tile_counts(to_display(seg[:, :, k] == v), to_display(fat_map[:, :, k]), side_of[v])
    tiles[v] = dict(total=tot, fat=ft)

def pct(fat, total):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(np.nan_to_num(total) > 0, 100 * fat / total, np.nan)

# --- the tiles drawn on the middle slice: one colour per tile position, hover gives muscle and tile
k_mid = order[n_slices // 2]
rgba = np.zeros(to_rgb(k_mid).shape[:2] + (4,), dtype=np.uint8)
names = np.full(rgba.shape[:2], "", dtype=object)
for v in present:
    idx = tile_index(to_display(seg[:, :, k_mid] == v)[crop], side_of[v])
    for t in range(N_AP * N_ML):
        m = idx == t
        rgba[m] = [int(TILE_COLORS[t][i:i + 2], 16) for i in (1, 3, 5)] + [255]
        names[m] = f"{display_name(v)} · {TILE_NAMES[t]}"
fig = go.Figure()
fig.add_trace(go.Image(z=to_rgb(k_mid), hoverinfo="skip"))
fig.add_trace(go.Image(z=rgba, colormodel="rgba", opacity=0.65, text=names, hovertemplate="%{text}<extra></extra>"))
for t in range(N_AP * N_ML):                          # legend entries
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(color=TILE_COLORS[t], size=12, symbol="square"),
                             name=TILE_NAMES[t], hoverinfo="skip"))
fig.update_xaxes(visible=False); fig.update_yaxes(visible=False, scaleanchor="x", autorange="reversed")
fig.update_layout(height=int(panel_height(1, fig_width=700)) + 60, width=800, margin=dict(l=10, r=10, t=36, b=10),
                  paper_bgcolor=CLEAR, plot_bgcolor=CLEAR, title=dict(text=f"slice {k_mid} · six tiles per muscle", font=dict(size=13)),
                  legend=dict(bgcolor="rgba(255,255,255,0.8)", font=dict(size=11)))
fig.show()

In [ ]:
# --- six tiles per muscle, all slices pooled
zmax_t = max(50, float(np.nanmax([pct(t["fat"].sum(0), np.nansum(t["total"], 0)) for t in tiles.values()])))
sides = ["Left", "Right"]
fig = make_subplots(rows=len(sides), cols=len(TARGETS), horizontal_spacing=0.08, vertical_spacing=0.16,
                    subplot_titles=[f"{side} {st}" for side in sides for st in TARGETS])
for r, side in enumerate(sides, start=1):
    for c, st in enumerate(TARGETS, start=1):
        if (st, side) not in muscles.index:
            continue
        v = muscles.loc[(st, side), "label"]
        z = pct(tiles[v]["fat"].sum(0), np.nansum(tiles[v]["total"], 0))
        fig.add_trace(go.Heatmap(z=z, x=ML_NAMES, y=AP_NAMES, zmin=0, zmax=zmax_t, colorscale="Inferno",
                                 text=z, texttemplate="%{text:.0f} %", textfont=dict(size=13),
                                 showscale=(r == 1 and c == len(TARGETS)), colorbar=dict(title="fat (%)", thickness=14, len=0.9),
                                 hovertemplate="%{y} · %{x}<br>fat %{z:.1f} %<extra>" + f"{side} {st}</extra>", xgap=2, ygap=2),
                      row=r, col=c)
        fig.update_yaxes(autorange="reversed", row=r, col=c)
fig.update_layout(height=520, margin=dict(l=10, r=10, t=50, b=10), paper_bgcolor=CLEAR, plot_bgcolor=CLEAR)
fig.show()

### Tiles along the muscle, superior to inferior

The same six tiles, now per slice. Each column is one slice, placed at its normalised position along the imaged muscle (superior to inferior); each row is one tile, anterior to posterior and medial to lateral. The colour and the number are the fat percentage of that tile on that slice.

In [ ]:
x_labels = [f"{p:.0f} %" for p in x_slices]
x_labels[0], x_labels[-1] = "superior", "inferior"
zmax_s = max(50, float(np.nanmax([pct(t["fat"], t["total"]) for t in tiles.values()])))
fig = make_subplots(rows=len(sides), cols=len(TARGETS), horizontal_spacing=0.08, vertical_spacing=0.16,
                    subplot_titles=[f"{side} {st}" for side in sides for st in TARGETS])
for r, side in enumerate(sides, start=1):
    for c, st in enumerate(TARGETS, start=1):
        if (st, side) not in muscles.index:
            continue
        v = muscles.loc[(st, side), "label"]
        z = pct(tiles[v]["fat"], tiles[v]["total"]).reshape(n_slices, N_AP * N_ML).T      # rows = tiles, cols = slices
        fig.add_trace(go.Heatmap(z=z, x=x_labels, y=TILE_NAMES, zmin=0, zmax=zmax_s, colorscale="Inferno",
                                 text=z, texttemplate="%{text:.0f}", textfont=dict(size=11),
                                 showscale=(r == 1 and c == len(TARGETS)), colorbar=dict(title="fat (%)", thickness=14, len=0.9),
                                 hovertemplate="length %{x} · %{y}<br>fat %{z:.1f} %<extra>" + f"{side} {st}</extra>", xgap=2, ygap=2),
                      row=r, col=c)
        fig.update_yaxes(autorange="reversed", row=r, col=c)
        fig.update_xaxes(title_text="Length" if r == len(sides) else None, row=r, col=c)
fig.update_layout(height=620, margin=dict(l=10, r=10, t=50, b=10), paper_bgcolor=CLEAR, plot_bgcolor=CLEAR)
fig.show()

## Example 2 · Thigh Dixon MRI

The lumbar example estimated fat indirectly, by clustering T2 intensities. With a **Dixon** acquisition the scanner separates the water and fat signal, and the **proton-density fat fraction** of every voxel is simply fat / (fat + water). This image is that fat-fraction map, acquired during the hands-on imaging session of this workshop: both thighs, 26 axial slices, voxel values 0–1000 for 0–100 %. Muscle is dark, fat is bright, and the whole-body model is contrast-agnostic, so it segments this image just as it did the T2-weighted spine. The file is stored in `data/MuscleMap/source-data/thigh/`.

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;">If <code>outputs/thigh/</code> already contains the segmentation, the MuscleMap cell below only shows the command and loads it. Otherwise it runs MuscleMap: 26 large slices, so allow 15–30 minutes on a CPU.</div>

In [ ]:
thigh_img = nib.as_closest_canonical(nib.load(thigh_image_path))   # RAS: x to the right, y to anterior, z to superior
ff_t = np.asanyarray(thigh_img.dataobj).astype(np.float32) / 10.0      # fat fraction in %
X_t, Y_t, Z_t = ff_t.shape
voxel_t = tuple(float(v) for v in thigh_img.header.get_zooms())
print("shape", ff_t.shape, "| voxel size (mm)", tuple(round(v, 2) for v in voxel_t), "| field of view (cm)", tuple(round(n * v / 10, 1) for n, v in zip(ff_t.shape, voxel_t)))

def thigh_display(a2d):
    """Axial slice (x, y) -> image with anterior at the top and the patient's right on the viewer's left."""
    return np.flipud(a2d[::-1].T)

# Plotly draws heatmaps above images, so the fat fraction is rendered as a colour image itself (inferno, 0-100 %)
from matplotlib import colormaps
inferno = colormaps["inferno"]
def ff_rgb(a2d):
    return (inferno(np.clip(a2d / 100.0, 0, 1))[..., :3] * 255).astype(np.uint8)

def add_ff_colorbar(fig):
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", showlegend=False, hoverinfo="skip",
                             marker=dict(colorscale="Inferno", cmin=0, cmax=100, color=[0], showscale=True,
                                         colorbar=dict(title="fat fraction (%)", thickness=14, len=0.8))), row=1, col=1)

# Display-only body mask: the fat fraction is pure noise outside the body (it averages to ~50 %), so hide it
from scipy.ndimage import median_filter, binary_fill_holes, binary_closing, binary_dilation, binary_opening, label as cc_label
def body_mask_2d(sl):
    med = median_filter(sl, size=9)
    t = (med < 30) | (med > 70)                      # tissue is either muscle (low) or fat (high); noise sits in between
    t = binary_opening(t, iterations=2)
    t = binary_closing(t, iterations=6)
    t = binary_fill_holes(t)
    lab, n = cc_label(t)
    if n == 0:
        return t
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    keep = [i for i in np.argsort(sizes)[::-1][:2] if sizes[i] > 5000]   # the two thighs
    return binary_dilation(np.isin(lab, keep), iterations=3)

# Four axial levels from distal to proximal, cropped to the body so the panels use their pixels well
levels = [int(Z_t * f) for f in (0.2, 0.4, 0.6, 0.8)]
level_names = ["distal", "lower mid", "upper mid", "proximal"]
masks = {k: body_mask_2d(ff_t[:, :, k]) for k in levels}
xs, ys = np.nonzero(np.any([masks[k] for k in levels], axis=0))
pad = 20
crop = (slice(max(xs.min() - pad, 0), xs.max() + pad), slice(max(ys.min() - pad, 0), ys.max() + pad))
def ff_panel(k):
    return thigh_display(np.where(masks[k], ff_t[:, :, k], 0)[crop])

fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.02, vertical_spacing=0.06, subplot_titles=[f"{n} thigh (slice {k})" for n, k in zip(level_names, levels)])
for i, k in enumerate(levels):
    r, c = divmod(i, 2)
    ff2d = ff_panel(k)
    fig.add_trace(go.Image(z=ff_rgb(ff2d), text=np.char.mod("fat fraction %.0f %%", ff2d), hovertemplate="%{text}<extra></extra>"), row=r + 1, col=c + 1)
    fig.update_xaxes(visible=False, row=r + 1, col=c + 1)
    fig.update_yaxes(visible=False, scaleanchor=f"x{i + 1 if i else ''}", autorange="reversed", row=r + 1, col=c + 1)
add_ff_colorbar(fig)
fig.update_layout(height=560, margin=dict(l=10, r=70, t=36, b=10), paper_bgcolor=CLEAR, plot_bgcolor=CLEAR)
fig.show()

### Segment the thighs

Same command as in step 2, only the image and the output folder differ.

In [ ]:
cmd = [mm_segment, "-i", str(thigh_image_path), "-r", "wholebody", "-o", str(thigh_output_dir)]
thigh_seg_path = thigh_output_dir / thigh_image_path.name.replace(".nii.gz", "_dseg.nii.gz")
display(HTML('<div style="font-family:ui-monospace,Menlo,Consolas,monospace;background:#17171a;border-radius:8px;padding:10px 14px;color:#e6e6e6;">'
             '<span style="color:#8a8a93;">$</span> mm_segment ' + " ".join(cmd[1:]) + "</div>"))

if thigh_seg_path.exists():
    display(HTML('<div style="color:#6e7781;margin:8px 0;">Precomputed segmentation found in <code>outputs/thigh/</code>, skipping the run. Delete the file to run MuscleMap yourself.</div>'))
else:
    lines, t0 = [], time.time()
    live = display(log_box([], "MuscleMap log - starting"), display_id=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        lines.append(line.rstrip("\n"))
        live.update(log_box(lines, f"MuscleMap log - running {time.time() - t0:.0f} s"))
    proc.wait()
    live.update(log_box(lines, f"MuscleMap log - {len(lines)} lines - {time.time() - t0:.0f} s"))
status_table([("Segmentation", thigh_seg_path.exists(), str(thigh_seg_path))])

### Fat fraction with the segmentation on top

The same four axial levels: the fat-fraction map underneath (dark = muscle, yellow = fat) and the MuscleMap labels on top. Hover for the structure name, use the slider to fade the overlay in and out.

In [ ]:
thigh_seg = np.asanyarray(nib.as_closest_canonical(nib.load(thigh_seg_path)).dataobj).astype(np.int32)
thigh_labels = [int(v) for v in np.unique(thigh_seg) if v != 0]
thigh_colors = {v: color_for(v) for v in thigh_labels}
print(f"{len(thigh_labels)} structures:", ", ".join(display_name(v) for v in thigh_labels))

def seg_rgba(seg2d, cmap, names_of):
    rgba = np.zeros(seg2d.shape + (4,), dtype=np.uint8)
    names = np.full(seg2d.shape, "", dtype=object)
    for v, c in cmap.items():
        m = seg2d == v
        if m.any():
            rgba[m] = [int(c[i:i + 2], 16) for i in (1, 3, 5)] + [255]
            names[m] = names_of(v)
    return rgba, names

fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.02, vertical_spacing=0.06, subplot_titles=[f"{n} thigh (slice {k})" for n, k in zip(level_names, levels)])
overlay_idx = []
for i, k in enumerate(levels):
    r, c = divmod(i, 2)
    ff2d = ff_panel(k)
    rgba, names = seg_rgba(thigh_display(thigh_seg[:, :, k][crop]), thigh_colors, display_name)
    fig.add_trace(go.Image(z=ff_rgb(ff2d), text=np.char.mod("fat fraction %.0f %%", ff2d), hovertemplate="%{text}<extra></extra>"), row=r + 1, col=c + 1)
    fig.add_trace(go.Image(z=rgba, colormodel="rgba", opacity=0.45, text=names, hovertemplate="%{text}<extra></extra>"), row=r + 1, col=c + 1)
    overlay_idx.append(len(fig.data) - 1)
    fig.update_xaxes(visible=False, row=r + 1, col=c + 1)
    fig.update_yaxes(visible=False, scaleanchor=f"x{i + 1 if i else ''}", autorange="reversed", row=r + 1, col=c + 1)
add_ff_colorbar(fig)
steps = [dict(method="restyle", label=f"{a:.1f}", args=[{"opacity": a}, overlay_idx]) for a in np.arange(0, 1.01, 0.1)]
fig.update_layout(height=640, margin=dict(l=10, r=70, t=36, b=90), paper_bgcolor=CLEAR, plot_bgcolor=CLEAR,
                  sliders=[dict(active=4, steps=steps, currentvalue=dict(prefix="segmentation opacity "), pad=dict(t=10), **SLIDER)])
fig.show()

### Volume and fat fraction per muscle

On a fat-fraction map no clustering is needed: `mm_extract_metrics -m average` simply averages the voxel values inside every label. It reports the mean intensity (here 0–1000, so divided by 10 for %) and the volume. The whole-body model also labels bone (femur, patella, tibia, fibula, ilium, sacrum); those are left out of the table, which lists muscles only.

In [ ]:
cmd = [mm_extract_metrics, "-m", "average", "-r", "wholebody", "-i", str(thigh_image_path), "-s", str(thigh_seg_path), "-o", str(thigh_output_dir)]
thigh_csv = thigh_output_dir / thigh_image_path.name.replace(".nii.gz", "_average_results.csv")
display(HTML('<div style="font-family:ui-monospace,Menlo,Consolas,monospace;background:#17171a;border-radius:8px;padding:10px 14px;color:#e6e6e6;">'
             '<span style="color:#8a8a93;">$</span> mm_extract_metrics ' + " ".join(cmd[1:]) + "</div>"))

if thigh_csv.exists():
    display(HTML('<div style="color:#6e7781;margin:8px 0;">Precomputed results found in <code>outputs/thigh/</code>, skipping the run.</div>'))
else:
    lines, t0 = [], time.time()
    live = display(log_box([], "MuscleMap log - starting"), display_id=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        lines.append(line.rstrip("\n"))
        live.update(log_box(lines, f"MuscleMap log - running {time.time() - t0:.0f} s"))
    proc.wait()
    live.update(log_box(lines, f"MuscleMap log - {len(lines)} lines - {time.time() - t0:.0f} s"))
status_table([("Results", thigh_csv.exists(), str(thigh_csv))])

In [ ]:
BONES = ("Femur", "Ilium", "Sacrum", "Tibia", "Fibula", "Patella")
muscle_labels = [v for v in thigh_labels if not any(b in structure_of(v) for b in BONES)]

res = pd.read_csv(thigh_csv)
res = res[res["Label"].isin(muscle_labels)].copy()
res["Fat fraction (%)"] = res["Average Intensity"] / 10.0          # image values are 0-1000 for 0-100 %
res["structure"] = res["Label"].map(structure_of)
res["side"] = res["Label"].map(lambda v: "Left" if v % 10 == 1 else "Right" if v % 10 == 2 else "Midline")
order_t = sorted(res["structure"].unique(), key=lambda st: res.loc[res.structure == st, "Label"].min())
table_t = pd.concat({"Volume (ml)": res.pivot_table(index="structure", columns="side", values="Volume (ml)"),
                     "Fat fraction (%)": res.pivot_table(index="structure", columns="side", values="Fat fraction (%)")}, axis=1).reindex(order_t)
swatch_t = {st: thigh_colors[int(res.loc[res.structure == st, "Label"].min())] for st in table_t.index}

styled = (table_t.style
          .format("{:.1f}", na_rep="–")
          .bar(subset=[c for c in table_t.columns if c[0] == "Fat fraction (%)"], color="#f9731666", vmin=0)
          .apply(lambda col: [f"border-left: 10px solid {swatch_t[i]}" for i in col.index], subset=[table_t.columns[0]])
          .set_caption(f"{len(muscle_labels)} muscles ({len(thigh_labels) - len(muscle_labels)} bone labels left out) · volume and Dixon fat fraction, left and right")
          .set_table_styles([{"selector": "caption", "props": "caption-side: top; color: #6e7781; padding-bottom: 6px;"}]))
display(styled)

## Where to go from here

* `mm_extract_metrics -m dixon` takes separate fat and water images; `-m average` also works on CT.
* `mm_register_to_template` brings segmentations into a common template space for group analyses.
* All labels, models and documentation: https://musclemap.github.io/MuscleMap/

<div style="border:1px solid #000000;border-left:5px solid #f97316;background:#ffffff;border-radius:0.5rem;padding:10px 16px;margin:10px 0;color:#111827;font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;">Always look at the segmentation before you trust the numbers. MuscleMap recommends checking and, where needed, correcting segmentations in <a href="https://www.slicer.org/" style="color:#0550ae;">3D Slicer</a> or <a href="http://www.itksnap.org/" style="color:#0550ae;">ITK-SNAP</a>.</div>